Optimizing Model Parameters
===========================

Now that we have a model and data it\'s time to train, validate and test
our model by optimizing its parameters on our data. Training a model is
an iterative process; in each iteration the model makes a guess about
the output, calculates the error in its guess (*loss*), collects the
derivatives of the error with respect to its parameters (as we saw in
the [previous section](autogradqs_tutorial.html)), and **optimizes**
these parameters using gradient descent. For a more detailed walkthrough
of this process, check out this video on [backpropagation from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).

Prerequisite Code
-----------------

We load the code from the previous sections on [Datasets &
DataLoaders](data_tutorial.html) and [Build
Model](buildmodel_tutorial.html).


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

Hyperparameters
===============

Hyperparameters are adjustable parameters that let you control the model
optimization process. Different hyperparameter values can impact model
training and convergence rates ([read
more](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)
about hyperparameter tuning)

We define the following hyperparameters for training:

-   **Number of Epochs** - the number of times to iterate over the
    dataset
-   **Batch Size** - the number of data samples propagated through the
    network before the parameters are updated
-   **Learning Rate** - how much to update models parameters at each
    batch/epoch. Smaller values yield slow learning speed, while large
    values may result in unpredictable behavior during training.


In [2]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

Optimization Loop
=================

Once we set our hyperparameters, we can then train and optimize our
model with an optimization loop. Each iteration of the optimization loop
is called an **epoch**.

Each epoch consists of two main parts:

-   **The Train Loop** - iterate over the training dataset and try to
    converge to optimal parameters.
-   **The Validation/Test Loop** - iterate over the test dataset to
    check if model performance is improving.

Let\'s briefly familiarize ourselves with some of the concepts used in
the training loop. Jump ahead to see the
`full-impl-label`{.interpreted-text role="ref"} of the optimization
loop.

Loss Function
-------------

When presented with some training data, our untrained network is likely
not to give the correct answer. **Loss function** measures the degree of
dissimilarity of obtained result to the target value, and it is the loss
function that we want to minimize during training. To calculate the loss
we make a prediction using the inputs of our given data sample and
compare it against the true data label value.

Common loss functions include
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) for regression tasks, and
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) for classification.
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
combines `nn.LogSoftmax` and `nn.NLLLoss`.

We pass our model\'s output logits to `nn.CrossEntropyLoss`, which will
normalize the logits and compute the prediction error.


In [3]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

Optimizer
=========

Optimization is the process of adjusting model parameters to reduce
model error in each training step. **Optimization algorithms** define
how this process is performed (in this example we use Stochastic
Gradient Descent). All optimization logic is encapsulated in the
`optimizer` object. Here, we use the SGD optimizer; additionally, there
are many [different
optimizers](https://pytorch.org/docs/stable/optim.html) available in
PyTorch such as ADAM and RMSProp, that work better for different kinds
of models and data.

We initialize the optimizer by registering the model\'s parameters that
need to be trained, and passing in the learning rate hyperparameter.


In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Inside the training loop, optimization happens in three steps:

-   Call `optimizer.zero_grad()` to reset the gradients of model
    parameters. Gradients by default add up; to prevent double-counting,
    we explicitly zero them at each iteration.
-   Backpropagate the prediction loss with a call to `loss.backward()`.
    PyTorch deposits the gradients of the loss w.r.t. each parameter.
-   Once we have our gradients, we call `optimizer.step()` to adjust the
    parameters by the gradients collected in the backward pass.


Full Implementation {#full-impl-label}
===================

We define `train_loop` that loops over our optimization code, and
`test_loop` that evaluates the model\'s performance against our test
data.


In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

We initialize the loss function and optimizer, and pass it to
`train_loop` and `test_loop`. Feel free to increase the number of epochs
to track the model\'s improving performance.

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.289938  [   64/60000]
loss: 2.282900  [ 6464/60000]
loss: 2.267840  [12864/60000]
loss: 2.273341  [19264/60000]
loss: 2.244355  [25664/60000]
loss: 2.221175  [32064/60000]
loss: 2.229331  [38464/60000]
loss: 2.192156  [44864/60000]
loss: 2.189125  [51264/60000]
loss: 2.174124  [57664/60000]
Test Error: 
 Accuracy: 42.9%, Avg loss: 2.155902 

Epoch 2
-------------------------------
loss: 2.156036  [   64/60000]
loss: 2.149688  [ 6464/60000]
loss: 2.090609  [12864/60000]
loss: 2.122767  [19264/60000]
loss: 2.056803  [25664/60000]
loss: 2.001243  [32064/60000]
loss: 2.035389  [38464/60000]
loss: 1.950509  [44864/60000]
loss: 1.964569  [51264/60000]
loss: 1.909964  [57664/60000]
Test Error: 
 Accuracy: 48.8%, Avg loss: 1.891844 

Epoch 3
-------------------------------
loss: 1.914493  [   64/60000]
loss: 1.888474  [ 6464/60000]
loss: 1.769354  [12864/60000]
loss: 1.831921  [19264/60000]
loss: 1.706769  [25664/60000]
loss: 1.666691  [32064/600

Summary
=======

- Training a model is iterative: each **epoch**, the model predicts, computes a **loss**, backpropagates gradients via `loss.backward()`, and updates parameters via gradient descent.
- **Hyperparameters** control the training process itself (not learned from data): number of epochs, batch size, and learning rate. Learning rate too small slows convergence; too large can make training unstable.
- The **loss function** (e.g. `nn.CrossEntropyLoss`, `nn.MSELoss`, `nn.NLLLoss`) measures how far the model's predictions are from the target — this is the value optimization tries to minimize.
- The **optimizer** (e.g. `torch.optim.SGD`, or alternatives like Adam/RMSProp) encapsulates the update rule; it's initialized with the model's parameters and the learning rate.
- Each training step follows the same three-call pattern:
  1. `optimizer.zero_grad()` — clear old gradients (they accumulate by default).
  2. `loss.backward()` — compute gradients of the loss w.r.t. each parameter.
  3. `optimizer.step()` — update parameters using those gradients.
- `model.train()` and `model.eval()` toggle behavior of layers like dropout/batch norm between training and evaluation.
- The **test/validation loop** runs under `torch.no_grad()` (no gradient tracking needed) to check generalization performance without updating weights.
- Over successive epochs in the example, training loss decreased and test accuracy rose (from ~43% to ~71% after 10 epochs), demonstrating the optimization loop working as intended.
